In [5]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

AttributeError: partially initialized module 'pandas' has no attribute '_pandas_parser_CAPI' (most likely due to a circular import)

In [7]:
!wget https://archive.ics.uci.edu/static/public/222/bank+marketing.zip
!unzip bank+marketing.zip
!unzip bank.zip

--2024-10-17 00:54:44--  https://archive.ics.uci.edu/static/public/222/bank+marketing.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘bank+marketing.zip’

bank+marketing.zip      [   <=>              ] 999.85K  2.14MB/s    in 0.5s    

2024-10-17 00:54:44 (2.14 MB/s) - ‘bank+marketing.zip’ saved [1023843]

Archive:  bank+marketing.zip
 extracting: bank.zip                
 extracting: bank-additional.zip     
Archive:  bank.zip
  inflating: bank-full.csv           
  inflating: bank-names.txt          
  inflating: bank.csv                


In [8]:
bank_data = pd.read_csv('bank-full.csv', sep=';')
bank_data.head()

NameError: name 'pd' is not defined

In [ ]:
bank_data.columns

In [ ]:
bank_data_columns = ['age','job','marital','education','balance','housing','loan','contact','day','month','duration','campaign','pdays','previous','poutcome','y']
bank_data = bank_data[bank_data_columns]
bank_data.head()

In [ ]:
print('Question 1. Mode for the column `education`')
print()
print(bank_data.education.groupby(bank_data.education).count().sort_values(ascending=False).head(1))

In [ ]:
bank_data_numerical_col = bank_data.columns[bank_data.dtypes == 'int64']
bank_data[bank_data_numerical_col]

In [ ]:
bank_data_corr_matrix = bank_data[bank_data_numerical_col].corr()
print()
print("Matrix correlation")
print(bank_data_corr_matrix)
bank_data_corr_list =  [['age and balance',bank_data_corr_matrix['age']['balance']]]
bank_data_corr_list += [['day and campaing',bank_data_corr_matrix['day']['campaign']]]
bank_data_corr_list += [['day and pdays',bank_data_corr_matrix['day']['pdays']]]
bank_data_corr_list += [['pdays and previous',bank_data_corr_matrix['pdays']['previous']]]
print()
print("Question 2. Two features with the biggest correlation")
print()
bank_data_corr_list.sort(reverse=True)
bank_data_corr_list[0]

In [9]:
bank_data_copy = bank_data.copy()

In [ ]:
bank_data_copy.y = bank_data_copy.y.map({'yes': 1, 'no': 0})
bank_data_copy.head()

In [ ]:
from sklearn.model_selection import train_test_split

bd_train_full, bd_test = train_test_split(bank_data_copy, test_size=0.2, random_state=42)
bd_train, bd_val = train_test_split(bd_train_full, test_size=0.25, random_state=11)

len(bank_data_copy),len(bd_train)+len(bd_val)+len(bd_test)

y_train = bd_train.y.values
y_val = bd_val.y.values
y_test = bd_test.y.values

del bd_train['y']
del bd_val['y']
del bd_test['y']

bd_train.head()



In [ ]:
from sklearn.metrics import mutual_info_score

bank_data_categorical_col = bd_train.columns[bd_train.dtypes == 'object']
bank_data_categorical_col

def mutual_info_y_score(series):
    return round(mutual_info_score(series, y_train),2)

bd_mi = bd_train[bank_data_categorical_col].apply(mutual_info_y_score)
bd_mi = bd_mi.sort_values(ascending=False)

print('Question 3. Variable with the biggest mutual information score')
print()
bd_mi

In [ ]:
from sklearn.feature_extraction import DictVectorizer

bd_train.reset_index(drop=True, inplace=True)
bd_val.reset_index(drop=True, inplace=True)
bd_test.reset_index(drop=True, inplace=True)

bd_cols = bank_data_categorical_col.tolist() + bank_data_numerical_col.tolist()

train_dict = bd_train[bd_cols].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)

X_train.shape

X_train[0]

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

In [ ]:
val_dict = bd_val[bd_cols].to_dict(orient='records')
X_val = dv.transform(val_dict)

y_pred = model.predict_proba(X_val)[:, 1]

model_accuracy = (y_val == (y_pred >= 0.5)).mean()
model_accuracy

print('Question 4. Accuracy on the validation dataset')
print()
print(round(model_accuracy,2))

In [ ]:
model_accuracy_diff = []

for col in bd_train.columns:
    print('Train without column: ',col)
    print()
    bd_train_copy = bd_train.copy()
    bd_val_copy = bd_val.copy()

    bd_train_copy.drop(col, axis=1, inplace=True)
    bd_val_copy.drop(col, axis=1, inplace=True)

    train_dict = bd_train_copy[bd_train_copy.columns].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(train_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    val_dict = bd_val_copy[bd_val_copy.columns].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    y_pred = model.predict_proba(X_val)[:, 1]

    model_accuracy_col = (y_val == (y_pred >= 0.5)).mean()

    print("Model accuracy all columns is:",round(model_accuracy,5))
    print("Model accuracy without column", col, "is:",round(model_accuracy_col,5))
    print("Diff:", round(model_accuracy - model_accuracy_col,5))
    print()

    if col in ['age','balance','marital','previous']:
      model_accuracy_diff.append([col,round(model_accuracy - model_accuracy_col,5)])

print('Question 5. Feature with the smallest difference')
print()
print(model_accuracy_diff)


In [ ]:
model_accuracy_array=[]

train_dict = bd_train[bd_train.columns].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)

val_dict = bd_val[bd_val.columns].to_dict(orient='records')
X_val = dv.transform(val_dict)

for r in [0.01,0.1,1,10,100]:
  model = LogisticRegression(solver='liblinear', C=r, max_iter=1000, random_state=42)
  model.fit(X_train, y_train)

  y_pred = model.predict_proba(X_val)[:, 1]

  model_accuracy_r = round((y_val == (y_pred >= 0.5)).mean(),3)
  model_accuracy_array.append([r,model_accuracy_r ])

print('Question 6. Smallest `C` that leads to the best accuracy on the validation set')
print()
print(model_accuracy_array)
